In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import optika
import furst

In [ ]:
astropy.visualization.quantity_support();

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="function 'sqrt' is not known")

In [ ]:
axis_channel = "channel"
axis_wavelength = "wavelength"
axis_field = ("field_x", "field_y")
axis_pupil = ("pupil_x", "pupil_y")

num_field = 15
num_pupil = 15

instrument = furst.instruments.design(num_wavelength=5)

# a separate draw for each channel
ones = na.ScalarArray(
    ndarray=np.ones(instrument.feed_optic.rowland_azimuth.shape[axis_channel]),
    axes=(axis_channel,),
)

instrument.field = na.Cartesian2dVectorStratifiedRandomSpace(
    start=-ones,
    stop=+ones,
    axis=na.Cartesian2dVectorArray(*axis_field),
    num=num_field,
    centers=True,
    seed=42,
)
instrument.pupil = na.Cartesian2dVectorStratifiedRandomSpace(
    start=-ones,
    stop=+ones,
    axis=na.Cartesian2dVectorArray(*axis_pupil),
    num=num_pupil,
    centers=True,
    seed=43,
)

system = instrument.system
axis_surface = system.axis_surface

In [ ]:
azimuth = instrument.feed_optic.rowland_azimuth
wavelength_min = instrument.wavelength_min.to(u.nm)
wavelength_max = instrument.wavelength_max.to(u.nm)

for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    print(
        f"channel {i}:"
        f" feed optic at {azimuth[index].ndarray:.2f},"
        f" {wavelength_min[index].ndarray:.1f} to {wavelength_max[index].ndarray:.1f}"
    )

In [ ]:
instrument_sparse = furst.instruments.design(
    num_wavelength=3,
    num_field=3,
    num_pupil=3,
)

fig, axs = plt.subplots(
    nrows=2,
    sharex=True,
    figsize=(9, 6),
    constrained_layout=True,
)
for ax, components in zip(axs, [("z", "x"), ("z", "y")]):
    instrument_sparse.system.plot(
        ax=ax,
        components=components,
        color="black",
        kwargs_rays=dict(
            color="tab:blue",
            linewidth=0.5,
        ),
    )
    ax.set_ylabel(f"${components[1]}$ ({ax.get_ylabel()})")
axs[1].set_xlabel(f"$z$ ({axs[1].get_xlabel()})")
axs[0].set_title("top view")
axs[1].set_title("side view")
axs[0].set_aspect("equal")

In [ ]:
raytrace = system.raytrace()
unvignetted = raytrace.outputs.unvignetted

index_surface = {
    "feed optic": 2,
    "grating": 3,
    "detector": 4,
}

fig, axs = plt.subplots(
    ncols=len(index_surface),
    figsize=(10, 4),
    constrained_layout=True,
)
for ax, name in zip(axs, index_surface):
    index = {axis_surface: index_surface[name]}
    surface = system.surfaces_all[index_surface[name]]
    position = surface.transformation.inverse(raytrace.outputs.position[index])
    na.plt.scatter(
        position.x,
        position.y,
        ax=ax,
        where=unvignetted[index],
        s=1,
    )
    wire = surface.aperture.wire()
    na.plt.plot(
        wire.x,
        wire.y,
        ax=ax,
        axis="wire",
        color="black",
    )
    ax.set_title(name)
    if name != "feed optic":
        ax.set_aspect("equal")

In [ ]:
rays = system.rayfunction_default.outputs

weight = rays.unvignetted.astype(float)
axes_disk = axis_field + axis_pupil

width_pixel = system.sensor.width_pixel

position = rays.position
position_mean = (position * weight).sum(axes_disk) / weight.sum(axes_disk)

dx = (position.x - position_mean.x) / width_pixel
dx = dx.to(u.dimensionless_unscaled) * u.pix

In [ ]:
index_center = {axis_wavelength: instrument.wavelength.shape[axis_wavelength] // 2}

lsf_2d = na.histogram2d(
    dx[index_center],
    position.y[index_center],
    bins=dict(lsf_x=21, lsf_y=21),
    axis=axes_disk,
    weights=weight[index_center],
    min=na.Cartesian2dVectorArray(-1 * u.pix, -8 * u.mm),
    max=na.Cartesian2dVectorArray(+1 * u.pix, +8 * u.mm),
)

fig, axs = na.plt.subplots(
    axis_cols=axis_channel,
    ncols=azimuth.shape[axis_channel],
    sharex=True,
    sharey=True,
    figsize=(10, 3.5),
    constrained_layout=True,
)
na.plt.pcolormesh(
    C=lsf_2d,
    ax=axs,
)
for i, ax in enumerate(axs.ndarray):
    ax.set_title(f"{azimuth[{axis_channel: i}].ndarray:.1f}")

In [ ]:
lsf = na.histogram(
    dx,
    bins=dict(lsf_x=41),
    axis=axes_disk,
    weights=weight,
    min=-1 * u.pix,
    max=+1 * u.pix,
)

fig, axs = na.plt.subplots(
    axis_cols=axis_channel,
    ncols=azimuth.shape[axis_channel],
    sharex=True,
    sharey=True,
    figsize=(10, 3),
    constrained_layout=True,
)
na.plt.stairs(
    lsf.inputs,
    lsf.outputs,
    ax=axs,
    axis="lsf_x",
)
for i, ax in enumerate(axs.ndarray):
    ax.set_title(f"{azimuth[{axis_channel: i}].ndarray:.1f}")
axs.ndarray[0].set_ylabel("rays");

In [ ]:
wavelength = instrument.wavelength_physical.to(u.nm)

variance_geometric = (np.square(dx) * weight).sum(axes_disk) / weight.sum(axes_disk)
variance_pixel = np.square(1 * u.pix) / 12
width = np.sqrt(variance_geometric + variance_pixel)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    na.plt.plot(
        wavelength[index],
        width[index],
        ax=ax,
        axis=axis_wavelength,
        marker="o",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
ax.set_ylabel(f"LSF width ({ax.get_ylabel()})")
ax.legend(title="feed optic azimuth");

In [ ]:
index_first = {axis_wavelength: 0}
index_last = {axis_wavelength: ~0}
dispersion = wavelength[index_last] - wavelength[index_first]
dispersion = dispersion / (position_mean.x[index_last] - position_mean.x[index_first])

wavelength_resolvable = 2 * width * (width_pixel / u.pix) * dispersion
resolving_power = (wavelength / wavelength_resolvable).to(u.dimensionless_unscaled)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    na.plt.plot(
        wavelength[index],
        resolving_power[index],
        ax=ax,
        axis=axis_wavelength,
        marker="o",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
ax.set_ylabel("resolving power")
ax.legend(title="feed optic azimuth");

In [ ]:
print(f"minimum resolving power: {resolving_power.min().ndarray:.0f}")
print(f"mean resolving power: {resolving_power.mean().ndarray:.0f}")
print(f"maximum resolving power: {resolving_power.max().ndarray:.0f}")

In [ ]:
fraction = unvignetted.mean(axes_disk + (axis_wavelength,))
names = [surface.name for surface in system.surfaces_all]

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    ax.plot(
        names,
        fraction[index].ndarray,
        marker="o",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.set_ylim(bottom=0)
ax.set_ylabel("fraction of rays surviving")
ax.legend(title="feed optic azimuth");

In [ ]:
index_grating = {axis_surface: index_surface["grating"]}
index_detector = {axis_surface: index_surface["detector"]}

position_detector = system.sensor.transformation.inverse(
    raytrace.outputs.position[index_detector]
)

half_height_detector = instrument.camera.sensor.num_pixel_active.y * width_pixel / 2
half_height_detector = half_height_detector.to(u.mm)

distribution_y = na.histogram(
    position_detector.y,
    bins=dict(lsf_y=41),
    axis=axes_disk + (axis_wavelength,),
    weights=unvignetted[index_grating].astype(float),
    min=-20 * u.mm,
    max=+20 * u.mm,
)

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
for i in range(azimuth.shape[axis_channel]):
    index = {axis_channel: i}
    na.plt.stairs(
        distribution_y.inputs,
        distribution_y.outputs[index],
        ax=ax,
        axis="lsf_y",
        label=f"{azimuth[index].ndarray:.1f}",
    )
ax.axvline(-half_height_detector, color="black", linestyle="--")
ax.axvline(+half_height_detector, color="black", linestyle="--")
ax.set_xlabel(f"height on the detector ({ax.get_xlabel()})")
ax.set_ylabel("rays")
ax.legend(title="feed optic azimuth");

In [ ]:
coating_design = furst.feed_optics.materials.coating_design()
coating_measured = furst.feed_optics.materials.coating_witness_measured()
angle_witness = furst.feed_optics.materials.angle_witness

measurement = coating_measured.efficiency_measured


# the reflectance of a coating at the given wavelength and angle
def reflectance(coating, wavelength, angle):
    rays = optika.rays.RayVectorArray(
        wavelength=wavelength,
        direction=na.Cartesian3dVectorArray(
            x=np.sin(angle),
            y=0,
            z=np.cos(angle),
        ),
    )
    return coating.efficiency(
        rays=rays,
        normal=na.Cartesian3dVectorArray(0, 0, -1),
    )


with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        figsize=(9, 3.5),
        constrained_layout=True,
    )
    for ax, (limit_min, limit_max) in zip(axs, [(115, 605), (118, 200)]):
        wavelength = na.linspace(
            start=limit_min,
            stop=limit_max,
            axis="wavelength",
            num=400,
        ) * u.nm
        na.plt.scatter(
            measurement.inputs.wavelength,
            measurement.outputs,
            ax=ax,
            s=10,
            color="black",
            label="witness sample",
        )
        na.plt.plot(
            wavelength,
            reflectance(coating_design, wavelength, angle_witness),
            ax=ax,
            axis="wavelength",
            color="tab:orange",
            label="model, matched to the vendor number at 121.6 nm",
        )
        # where the vendor's own published curve turns over, read off
        # the catalogue plot
        ax.scatter(
            [150, 177],
            [0.855, 0.780],
            marker="x",
            color="tab:red",
            label="vendor curve, published extrema",
        )
        ax.set_xlim(limit_min, limit_max)
        ax.set_xlabel(f"wavelength ({ax.get_xlabel()})")
    axs[0].set_ylabel("reflectance")
    axs[1].set_ylim(0.74, 0.88)

handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncols=3, fontsize=8);

In [ ]:
# the angles at which the feed optics are actually illuminated, taken
# from the raytrace above
index_feed = {axis_surface: index_surface["feed optic"]}
surface_feed = system.surfaces_all[index_surface["feed optic"]]

position_feed = surface_feed.transformation.inverse(
    raytrace.outputs.position[index_feed]
)
direction_feed = surface_feed.transformation.transformation_linear.inverse(
    raytrace.outputs.direction[index_feed]
)
cosine = np.abs(direction_feed @ surface_feed.sag.normal(position_feed))
angle_feed = np.arccos(np.clip(na.value(cosine), -1, 1)) * u.rad
angle_feed = angle_feed[unvignetted[index_feed]].to(u.deg)

print(f"feed optic angle of incidence: {angle_feed.min().ndarray:.1f}"
      f" to {angle_feed.max().ndarray:.1f}, mean {angle_feed.mean().ndarray:.1f}")
print(f"witness sample measured at:    {angle_witness}")

for angle in [angle_witness, angle_feed.mean().ndarray]:
    r = reflectance(coating_design, instrument.wavelength_physical, angle)
    print(f"  band-mean reflectance at {angle:.1f}: {r.mean().ndarray:.4f}")